## Data Loading and Initial Inspection

I downloaded the training dataset from the Kaggle competition:

`https://www.kaggle.com/competitions/DontGetKicked/data?select=training.csv`

I saved the file locally as:

`data/training.csv`

The target column is `IsBadBuy`, which represents a binary classification target:

- `0`: the car is not a bad buy
- `1`: the car is a bad buy

Before building any model, I inspected the dataset shape, previewed the first rows, checked column types and missing values, and reviewed the target distribution.

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("data/training.csv")

print("Shape:", df.shape)
display(df.head())

print("\nInfo:")
df.info()

print("\nTarget counts:")
print(df["IsBadBuy"].value_counts())

print("\nTarget distribution:")
print(df["IsBadBuy"].value_counts(normalize=True))

Shape: (72983, 34)


,RefId,IsBadBuy,PurchDate,Auction,VehYear,VehicleAge,Make,Model,Trim,SubModel,...,MMRCurrentRetailAveragePrice,MMRCurrentRetailCleanPrice,PRIMEUNIT,AUCGUART,BYRNO,VNZIP1,VNST,VehBCost,IsOnlineSale,WarrantyCost
0,1,0,12/7/2009,ADESA,2006,3,MAZDA,MAZDA3,i,4D SEDAN I,...,11597.0,12409.0,NaN,NaN,21973,33619,FL,7100.0,0,1113
1,2,0,12/7/2009,ADESA,2004,5,DODGE,1500 RAM PICKUP 2WD,ST,QUAD CAB 4.7L SLT,...,11374.0,12791.0,NaN,NaN,19638,33619,FL,7600.0,0,1053
2,3,0,12/7/2009,ADESA,2005,4,DODGE,STRATUS V6,SXT,4D SEDAN SXT FFV,...,7146.0,8702.0,NaN,NaN,19638,33619,FL,4900.0,0,1389
3,4,0,12/7/2009,ADESA,2004,5,DODGE,NEON,SXT,4D SEDAN,...,4375.0,5518.0,NaN,NaN,19638,33619,FL,4100.0,0,630
4,5,0,12/7/2009,ADESA,2005,4,FORD,FOCUS,ZX3,2D COUPE ZX3,...,6739.0,7911.0,NaN,NaN,19638,33619,FL,4000.0,0,1020



Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 72983 entries, 0 to 72982
Data columns (total 34 columns):
 #   Column                             Non-Null Count  Dtype  
---  ------                             --------------  -----  
 0   RefId                              72983 non-null  int64  
 1   IsBadBuy                           72983 non-null  int64  
 2   PurchDate                          72983 non-null  object 
 3   Auction                            72983 non-null  object 
 4   VehYear                            72983 non-null  int64  
 5   VehicleAge                         72983 non-null  int64  
 6   Make                               72983 non-null  object 
 7   Model                              72983 non-null  object 
 8   Trim                               70623 non-null  object 
 9   SubModel                           72975 non-null  object 
 10  Color                              72975 non-null  object 
 11  Transmission                       72974 non-nu

## Initial Observations

I loaded 72,983 rows and 34 columns.

The target column is `IsBadBuy`. The dataset is imbalanced:
- around 87.7% of the cars are not bad buys (`0`)
- around 12.3% of the cars are bad buys (`1`)

This means accuracy alone is not enough to evaluate the models. I will use ranking and classification metrics such as Gini, ROC AUC, Precision, Recall, and F1.

I also noticed that `PurchDate` is stored as an object, so I need to convert it to a datetime column before splitting the data by purchase date.

In [2]:
df["PurchDate"] = pd.to_datetime(df["PurchDate"])

print(df["PurchDate"].dtype)
print("Min date:", df["PurchDate"].min())
print("Max date:", df["PurchDate"].max())

datetime64[ns]
Min date: 2009-01-05 00:00:00
Max date: 2010-12-30 00:00:00


## Purchase Date Conversion

I converted `PurchDate` from object type to datetime.

The dataset covers purchases from `2009-01-05` to `2010-12-30`.

Since this is time-based data, I will split it by purchase date instead of using a random split. This helps avoid training the model on future data and testing it on the past.

In [3]:
df = df.sort_values("PurchDate").reset_index(drop=True)

n = len(df)

train_end = int(n * 0.6)
valid_end = int(n * 0.8)

train_df = df.iloc[:train_end].copy()
valid_df = df.iloc[train_end:valid_end].copy()
test_df = df.iloc[valid_end:].copy()

print("Train shape:", train_df.shape)
print("Valid shape:", valid_df.shape)
print("Test shape:", test_df.shape)

print("\nTrain date range:", train_df["PurchDate"].min(), "to", train_df["PurchDate"].max())
print("Valid date range:", valid_df["PurchDate"].min(), "to", valid_df["PurchDate"].max())
print("Test date range:", test_df["PurchDate"].min(), "to", test_df["PurchDate"].max())

print("\nTarget distribution:")
print("Train:")
print(train_df["IsBadBuy"].value_counts(normalize=True))
print("\nValid:")
print(valid_df["IsBadBuy"].value_counts(normalize=True))
print("\nTest:")
print(test_df["IsBadBuy"].value_counts(normalize=True))

Train shape: (43789, 34)
Valid shape: (14597, 34)
Test shape: (14597, 34)

Train date range: 2009-01-05 00:00:00 to 2010-03-29 00:00:00
Valid date range: 2010-03-29 00:00:00 to 2010-08-18 00:00:00
Test date range: 2010-08-18 00:00:00 to 2010-12-30 00:00:00

Target distribution:
Train:
IsBadBuy
0    0.878691
1    0.121309
Name: proportion, dtype: float64

Valid:
IsBadBuy
0    0.876618
1    0.123382
Name: proportion, dtype: float64

Test:
IsBadBuy
0    0.872371
1    0.127629
Name: proportion, dtype: float64


## Time-Based Data Split

I sorted the dataset by `PurchDate` and split it into three parts:

- 60% training data
- 20% validation data
- 20% test data

The training data contains older purchases, the validation data contains the next time period, and the test data contains the most recent purchases.

The target distribution is similar across the three splits, with around 12–13% bad buys in each part. This makes the split suitable for model training and evaluation.

In [4]:
target = "IsBadBuy"

drop_columns = ["IsBadBuy", "RefId", "PurchDate"]

X_train = train_df.drop(columns=drop_columns)
y_train = train_df[target]

X_valid = valid_df.drop(columns=drop_columns)
y_valid = valid_df[target]

X_test = test_df.drop(columns=drop_columns)
y_test = test_df[target]

print("X_train:", X_train.shape)
print("y_train:", y_train.shape)

print("X_valid:", X_valid.shape)
print("y_valid:", y_valid.shape)

print("X_test:", X_test.shape)
print("y_test:", y_test.shape)

X_train: (43789, 31)
y_train: (43789,)
X_valid: (14597, 31)
y_valid: (14597,)
X_test: (14597, 31)
y_test: (14597,)


## Feature and Target Separation

I separated the target column from the input features.

The target is:

`IsBadBuy`

I removed `IsBadBuy`, `RefId`, and `PurchDate` from the feature set:
- `IsBadBuy` is the target
- `RefId` is only an identifier
- `PurchDate` was used for time-based splitting

After this step, each dataset has 31 input features.

In [5]:
numeric_features = X_train.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_features = X_train.select_dtypes(include=["object"]).columns.tolist()

print("Numeric features:", len(numeric_features))
print(numeric_features)

print("\nCategorical features:", len(categorical_features))
print(categorical_features)

Numeric features: 17
['VehYear', 'VehicleAge', 'WheelTypeID', 'VehOdo', 'MMRAcquisitionAuctionAveragePrice', 'MMRAcquisitionAuctionCleanPrice', 'MMRAcquisitionRetailAveragePrice', 'MMRAcquisitonRetailCleanPrice', 'MMRCurrentAuctionAveragePrice', 'MMRCurrentAuctionCleanPrice', 'MMRCurrentRetailAveragePrice', 'MMRCurrentRetailCleanPrice', 'BYRNO', 'VNZIP1', 'VehBCost', 'IsOnlineSale', 'WarrantyCost']

Categorical features: 14
['Auction', 'Make', 'Model', 'Trim', 'SubModel', 'Color', 'Transmission', 'WheelType', 'Nationality', 'Size', 'TopThreeAmericanName', 'PRIMEUNIT', 'AUCGUART', 'VNST']


## Feature Types

I separated the input features into numeric and categorical columns.

The dataset contains:
- 17 numeric features
- 14 categorical features

Numeric features will need missing-value handling and scaling.
Categorical features will need missing-value handling and encoding before training the models.

In [6]:
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

## Baseline Model: Logistic Regression

I started with Logistic Regression as the first baseline model.

This model is suitable for binary classification because it predicts the probability of the positive class, which is `IsBadBuy = 1`.

I used the preprocessing pipeline before the model so that numeric and categorical features are handled correctly.

In [7]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

log_reg_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(max_iter=1000, random_state=21))
])

log_reg_model.fit(X_train, y_train)

valid_proba = log_reg_model.predict_proba(X_valid)[:, 1]

valid_auc = roc_auc_score(y_valid, valid_proba)
valid_gini = 2 * valid_auc - 1

print("Validation ROC AUC:", valid_auc)
print("Validation Gini:", valid_gini)


Validation ROC AUC: 0.6752639853618887
Validation Gini: 0.3505279707237774


## Logistic Regression Baseline

Logistic Regression was used as the first baseline model for the binary classification task.

The model was trained using the preprocessing pipeline, which handles missing values, scales numeric features, and encodes categorical features.

On the validation set, the model achieved:

- ROC AUC: `0.6753`
- Gini score: `0.3505`

The Gini score is above the required minimum of `0.15`, so this baseline model gives a reasonable starting point for comparison with the other classification algorithms.

## Gaussian Naive Bayes Baseline

Gaussian Naive Bayes was trained as the second baseline model.

This model uses probability-based assumptions to classify each car as a good or bad buy. Since `GaussianNB` requires dense input, a separate preprocessing pipeline was created with dense one-hot encoding for categorical features.

The model performance was evaluated on the validation set using ROC AUC and Gini score.

In [8]:
from sklearn.naive_bayes import GaussianNB
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

numeric_transformer_nb = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer_nb = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

preprocessor_nb = ColumnTransformer(transformers=[
    ("num", numeric_transformer_nb, numeric_features),
    ("cat", categorical_transformer_nb, categorical_features)
])

nb_model = Pipeline(steps=[
    ("preprocessor", preprocessor_nb),
    ("model", GaussianNB())
])

nb_model.fit(X_train, y_train)

valid_proba_nb = nb_model.predict_proba(X_valid)[:, 1]

valid_auc_nb = roc_auc_score(y_valid, valid_proba_nb)
valid_gini_nb = 2 * valid_auc_nb - 1

print("Validation ROC AUC:", valid_auc_nb)
print("Validation Gini:", valid_gini_nb)

Validation ROC AUC: 0.4918109299494792
Validation Gini: -0.01637814010104155


## Gaussian Naive Bayes Baseline Result

Gaussian Naive Bayes was trained and evaluated on the validation set.

The model achieved:

- ROC AUC: `0.4918`
- Gini score: `-0.0164`

This result is very weak because the ROC AUC is below `0.5`, which means the model performs worse than random ranking on the validation set.

Compared with Logistic Regression, Gaussian Naive Bayes is not a good baseline for this dataset.

## KNN Baseline

KNN was trained as the third baseline classification model.

This model predicts each car by looking at the nearest similar cars in the training data and using their labels for voting.

Since KNN depends on distance calculations, numeric features were scaled before training. The model was evaluated on the validation set using ROC AUC and Gini score.

In [9]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import roc_auc_score

knn_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", KNeighborsClassifier(n_neighbors=5))
])

knn_model.fit(X_train, y_train)

valid_proba_knn = knn_model.predict_proba(X_valid)[:, 1]

valid_auc_knn = roc_auc_score(y_valid, valid_proba_knn)
valid_gini_knn = 2 * valid_auc_knn - 1

print("Validation ROC AUC:", valid_auc_knn)
print("Validation Gini:", valid_gini_knn)

Validation ROC AUC: 0.5963649627460275
Validation Gini: 0.19272992549205492


## KNN Baseline Result

KNN was trained and evaluated on the validation set.

The model achieved:

- ROC AUC: `0.5964`
- Gini score: `0.1927`

The Gini score is above the required minimum of `0.15`, so KNN works as a valid baseline.

However, it performs worse than Logistic Regression, which achieved a higher validation Gini score.

In [10]:
results = pd.DataFrame({
    "model": [
        "Logistic Regression",
        "Gaussian Naive Bayes",
        "KNN"
    ],
    "validation_roc_auc": [
        valid_auc,
        valid_auc_nb,
        valid_auc_knn
    ],
    "validation_gini": [
        valid_gini,
        valid_gini_nb,
        valid_gini_knn
    ]
})

results.sort_values("validation_gini", ascending=False)

,model,validation_roc_auc,validation_gini
0,Logistic Regression,0.675264,0.350528
2,KNN,0.596365,0.192730
1,Gaussian Naive Bayes,0.491811,-0.016378


## Baseline Model Comparison Result

The three baseline models were compared on the validation set.

Logistic Regression achieved the best result with a validation Gini score of `0.3505`.

KNN also produced an acceptable result with a Gini score of `0.1927`, but it was weaker than Logistic Regression.

Gaussian Naive Bayes performed poorly with a negative Gini score, so it is not a good option for this dataset.

## Classification Metrics for the Best Baseline Model

Logistic Regression will be evaluated using additional classification metrics.

ROC AUC and Gini evaluate the quality of predicted probabilities, while Precision, Recall, and F1 evaluate the final class predictions.

This helps understand how well the model detects bad-buy cars after converting probabilities into class labels.

In [11]:
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score

valid_pred = log_reg_model.predict(X_valid)
valid_proba = log_reg_model.predict_proba(X_valid)[:, 1]

precision = precision_score(y_valid, valid_pred)
recall = recall_score(y_valid, valid_pred)
f1 = f1_score(y_valid, valid_pred)
roc_auc = roc_auc_score(y_valid, valid_proba)
gini = 2 * roc_auc - 1

print("Precision:", precision)
print("Recall:", recall)
print("F1:", f1)
print("ROC AUC:", roc_auc)
print("Gini:", gini)

Precision: 0.2702702702702703
Recall: 0.0277623542476402
F1: 0.050352467270896276
ROC AUC: 0.6752639853618887
Gini: 0.3505279707237774


## Logistic Regression Classification Metrics Result

Logistic Regression achieved:

- Precision: `0.2703`
- Recall: `0.0278`
- F1 score: `0.0504`
- ROC AUC: `0.6753`
- Gini score: `0.3505`

The ROC AUC and Gini scores show that the model can rank cars reasonably well by bad-buy risk.

However, the Recall and F1 score are very low. This means that using the default threshold of `0.5`, the model detects only a small number of actual bad-buy cars.

This happens because the dataset is imbalanced, and only about 12–13% of the cars are bad buys. The model is conservative and predicts most cars as class `0`.

## Threshold Tuning

The default classification threshold is `0.5`.

Since the dataset is imbalanced and the model has very low recall, different thresholds will be tested on the validation set.

The goal is to find a threshold that improves the balance between Precision and Recall, using F1 score as the main comparison metric.

In [12]:
from sklearn.metrics import precision_score, recall_score, f1_score

threshold_results = []

for threshold in np.arange(0.05, 0.51, 0.05):
    y_pred_threshold = (valid_proba >= threshold).astype(int)
    
    precision_t = precision_score(y_valid, y_pred_threshold, zero_division=0)
    recall_t = recall_score(y_valid, y_pred_threshold)
    f1_t = f1_score(y_valid, y_pred_threshold)
    
    threshold_results.append({
        "threshold": threshold,
        "precision": precision_t,
        "recall": recall_t,
        "f1": f1_t
    })

threshold_results_df = pd.DataFrame(threshold_results)

threshold_results_df.sort_values("f1", ascending=False)

,threshold,precision,recall,f1
2,0.15,0.204684,0.558023,0.299508
3,0.20,0.227953,0.397557,0.289761
1,0.10,0.171857,0.738479,0.278826
4,0.25,0.249495,0.274292,0.261307
0,0.05,0.141246,0.931705,0.245304
5,0.30,0.276800,0.192115,0.226811
6,0.35,0.293578,0.124375,0.174727
7,0.40,0.300830,0.080511,0.127026
8,0.45,0.313131,0.051638,0.088656
9,0.50,0.270270,0.027762,0.050352


## Threshold Tuning Result

Different classification thresholds were tested on the validation set.

The best threshold based on F1 score was `0.15`.

At this threshold, Logistic Regression achieved:

- Precision: `0.2047`
- Recall: `0.5580`
- F1 score: `0.2995`

This is a large improvement compared with the default threshold of `0.5`.

The lower threshold makes the model detect more bad-buy cars, which improves Recall. However, Precision becomes lower because the model also produces more false positives.

For this problem, improving Recall is important because missing a bad-buy car can be costly.

In [13]:
best_threshold = 0.15

valid_pred_best_threshold = (valid_proba >= best_threshold).astype(int)

precision_best = precision_score(y_valid, valid_pred_best_threshold, zero_division=0)
recall_best = recall_score(y_valid, valid_pred_best_threshold)
f1_best = f1_score(y_valid, valid_pred_best_threshold)

print("Best threshold:", best_threshold)
print("Precision:", precision_best)
print("Recall:", recall_best)
print("F1:", f1_best)

Best threshold: 0.15
Precision: 0.20468431771894094
Recall: 0.558023320377568
F1: 0.2995082700044703


## Selected Classification Threshold

The best threshold on the validation set was selected as `0.15`.

Using this threshold, Logistic Regression achieved:

- Precision: `0.2047`
- Recall: `0.5580`
- F1 score: `0.2995`

This threshold gives a better balance between Precision and Recall than the default threshold of `0.5`.

The model now detects more actual bad-buy cars, although this also increases the number of false positives.

## Feature Engineering

New features will be created to help the model capture useful relationships in the data.

The new features focus on:
- vehicle usage
- cost relationships
- market price differences
- warranty cost compared with vehicle cost

In [14]:
def add_features(data):
    data = data.copy()
    
    data["OdoPerYear"] = data["VehOdo"] / (data["VehicleAge"] + 1)
    data["WarrantyToCostRatio"] = data["WarrantyCost"] / (data["VehBCost"] + 1)
    
    data["AcquisitionAuctionSpread"] = (
        data["MMRAcquisitionAuctionCleanPrice"] 
        - data["MMRAcquisitionAuctionAveragePrice"]
    )
    
    data["CurrentAuctionSpread"] = (
        data["MMRCurrentAuctionCleanPrice"] 
        - data["MMRCurrentAuctionAveragePrice"]
    )
    
    data["RetailAuctionDiff"] = (
        data["MMRAcquisitionRetailAveragePrice"] 
        - data["MMRAcquisitionAuctionAveragePrice"]
    )
    
    data["CurrentRetailAuctionDiff"] = (
        data["MMRCurrentRetailAveragePrice"] 
        - data["MMRCurrentAuctionAveragePrice"]
    )
    
    data["CostToAuctionAvgRatio"] = (
        data["VehBCost"] / (data["MMRAcquisitionAuctionAveragePrice"] + 1)
    )
    
    return data

In [15]:
train_fe = add_features(train_df)
valid_fe = add_features(valid_df)
test_fe = add_features(test_df)

X_train_fe = train_fe.drop(columns=drop_columns)
y_train_fe = train_fe[target]

X_valid_fe = valid_fe.drop(columns=drop_columns)
y_valid_fe = valid_fe[target]

X_test_fe = test_fe.drop(columns=drop_columns)
y_test_fe = test_fe[target]

numeric_features_fe = X_train_fe.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_features_fe = X_train_fe.select_dtypes(include=["object"]).columns.tolist()

print("Numeric features:", len(numeric_features_fe))
print("Categorical features:", len(categorical_features_fe))

Numeric features: 24
Categorical features: 14


## Feature Engineering Result

Seven new numeric features were added to the dataset.

After feature engineering, the dataset contains:

- 24 numeric features
- 14 categorical features

These new features describe relationships between mileage, vehicle age, warranty cost, vehicle cost, and market price estimates.

In [16]:
numeric_transformer_fe = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer_fe = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor_fe = ColumnTransformer(transformers=[
    ("num", numeric_transformer_fe, numeric_features_fe),
    ("cat", categorical_transformer_fe, categorical_features_fe)
])

log_reg_fe_model = Pipeline(steps=[
    ("preprocessor", preprocessor_fe),
    ("model", LogisticRegression(max_iter=1000, random_state=21))
])

log_reg_fe_model.fit(X_train_fe, y_train_fe)

valid_proba_fe = log_reg_fe_model.predict_proba(X_valid_fe)[:, 1]

valid_auc_fe = roc_auc_score(y_valid_fe, valid_proba_fe)
valid_gini_fe = 2 * valid_auc_fe - 1

print("Validation ROC AUC:", valid_auc_fe)
print("Validation Gini:", valid_gini_fe)

Validation ROC AUC: 0.6750580457975571
Validation Gini: 0.35011609159511425


## Logistic Regression with Engineered Features Result

Logistic Regression was trained again after adding the engineered features.

The model achieved:

- ROC AUC: `0.6751`
- Gini score: `0.3501`

The result is almost the same as the original Logistic Regression baseline.

The engineered features did not improve the validation Gini score, so the original feature set remains slightly better for this model.

## Logistic Regression with Class Weighting

Since the dataset is imbalanced, class weighting will be tested.

The goal is to give more importance to the minority class, `IsBadBuy = 1`, during training.

In [17]:
log_reg_balanced_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(
        max_iter=1000,
        random_state=21,
        class_weight="balanced"
    ))
])

log_reg_balanced_model.fit(X_train, y_train)

valid_proba_balanced = log_reg_balanced_model.predict_proba(X_valid)[:, 1]

valid_auc_balanced = roc_auc_score(y_valid, valid_proba_balanced)
valid_gini_balanced = 2 * valid_auc_balanced - 1

print("Validation ROC AUC:", valid_auc_balanced)
print("Validation Gini:", valid_gini_balanced)

Validation ROC AUC: 0.6667652249045762
Validation Gini: 0.3335304498091525


## Logistic Regression with Class Weighting Result

Logistic Regression was trained with `class_weight="balanced"` to handle the imbalanced target distribution.

The model achieved:

- ROC AUC: `0.6668`
- Gini score: `0.3335`

This result is lower than the original Logistic Regression baseline, which achieved a Gini score of `0.3505`.

Therefore, class weighting did not improve the model performance on the validation set.

## Logistic Regression Regularization Tuning

Different values of `C` will be tested for Logistic Regression.

The goal is to find whether changing the regularization strength improves the validation Gini score.

In [ ]:
c_values = [0.01, 0.05, 0.1, 0.5, 1, 2, 5, 10]

c_results = []

for c in c_values:
    model = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("model", LogisticRegression(
            C=c,
            max_iter=1000,
            random_state=21
        ))
    ])
    
    model.fit(X_train, y_train)
    
    valid_proba_c = model.predict_proba(X_valid)[:, 1]
    valid_auc_c = roc_auc_score(y_valid, valid_proba_c)
    valid_gini_c = 2 * valid_auc_c - 1
    
    c_results.append({
        "C": c,
        "validation_roc_auc": valid_auc_c,
        "validation_gini": valid_gini_c
    })

c_results_df = pd.DataFrame(c_results)
c_results_df.sort_values("validation_gini", ascending=False)

## Logistic Regression Regularization Tuning Result

Different values of `C` were tested for Logistic Regression.

The best validation result was achieved with:

- `C = 0.05`
- ROC AUC: `0.6848`
- Gini score: `0.3697`

This is better than the original Logistic Regression baseline, which achieved a Gini score of `0.3505`.

A smaller value of `C` improved the model performance, which suggests that stronger regularization helps this model generalize better.

In [ ]:
best_log_reg_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(
        C=0.05,
        max_iter=1000,
        random_state=21
    ))
])

best_log_reg_model.fit(X_train, y_train)

best_valid_proba = best_log_reg_model.predict_proba(X_valid)[:, 1]

best_valid_auc = roc_auc_score(y_valid, best_valid_proba)
best_valid_gini = 2 * best_valid_auc - 1

print("Validation ROC AUC:", best_valid_auc)
print("Validation Gini:", best_valid_gini)

## Best Logistic Regression Model

The best Logistic Regression model was selected using validation Gini score.

The selected model uses:

- `C = 0.05`
- ROC AUC: `0.6848`
- Gini score: `0.3697`

This model performs better than the original Logistic Regression baseline, so it will be used as the strongest model so far.

In [ ]:
best_threshold_results = []

for threshold in np.arange(0.05, 0.51, 0.05):
    y_pred_threshold = (best_valid_proba >= threshold).astype(int)
    
    precision_t = precision_score(y_valid, y_pred_threshold, zero_division=0)
    recall_t = recall_score(y_valid, y_pred_threshold)
    f1_t = f1_score(y_valid, y_pred_threshold)
    
    best_threshold_results.append({
        "threshold": threshold,
        "precision": precision_t,
        "recall": recall_t,
        "f1": f1_t
    })

best_threshold_results_df = pd.DataFrame(best_threshold_results)
best_threshold_results_df.sort_values("f1", ascending=False)

## Threshold Tuning for the Best Logistic Regression Model

Different thresholds were tested again after tuning Logistic Regression.

The best F1 score was achieved with:

- Threshold: `0.15`
- Precision: `0.2086`
- Recall: `0.5730`
- F1 score: `0.3058`

Compared with the previous Logistic Regression model, the tuned model gives a slightly better F1 score and detects more bad-buy cars.

The threshold `0.15` will be used with the selected Logistic Regression model.

In [ ]:
final_threshold = 0.15

best_valid_pred = (best_valid_proba >= final_threshold).astype(int)

final_precision = precision_score(y_valid, best_valid_pred, zero_division=0)
final_recall = recall_score(y_valid, best_valid_pred)
final_f1 = f1_score(y_valid, best_valid_pred)

print("Final validation threshold:", final_threshold)
print("Precision:", final_precision)
print("Recall:", final_recall)
print("F1:", final_f1)
print("ROC AUC:", best_valid_auc)
print("Gini:", best_valid_gini)

## Final Validation Result

The selected model is Logistic Regression with `C = 0.05`.

The selected classification threshold is `0.15`.

On the validation set, the final model achieved:

- Precision: `0.2086`
- Recall: `0.5730`
- F1 score: `0.3058`
- ROC AUC: `0.6848`
- Gini score: `0.3697`

The model performs better after regularization tuning. The lower threshold improves Recall, which is useful because the main goal is to detect more bad-buy cars.

In [ ]:
test_proba = best_log_reg_model.predict_proba(X_test)[:, 1]
test_pred = (test_proba >= final_threshold).astype(int)

test_auc = roc_auc_score(y_test, test_proba)
test_gini = 2 * test_auc - 1
test_precision = precision_score(y_test, test_pred, zero_division=0)
test_recall = recall_score(y_test, test_pred)
test_f1 = f1_score(y_test, test_pred)

print("Test Precision:", test_precision)
print("Test Recall:", test_recall)
print("Test F1:", test_f1)
print("Test ROC AUC:", test_auc)
print("Test Gini:", test_gini)

## Final Test Evaluation

The final model was evaluated once on the test set.

The selected model is Logistic Regression with:

- `C = 0.05`
- classification threshold = `0.15`

On the test set, the model achieved:

- Precision: `0.2123`
- Recall: `0.6302`
- F1 score: `0.3176`
- ROC AUC: `0.6991`
- Gini score: `0.3982`

The test Gini score is higher than the validation Gini score, which means the model performed well on unseen future data.

The model also achieved better Recall on the test set, detecting around 63% of actual bad-buy cars. Precision is still low, which means the model produces many false positives, but this can be acceptable if missing bad-buy cars is more costly than reviewing extra cars.

In [ ]:
final_results = pd.DataFrame({
    "dataset": ["Validation", "Test"],
    "precision": [final_precision, test_precision],
    "recall": [final_recall, test_recall],
    "f1": [final_f1, test_f1],
    "roc_auc": [best_valid_auc, test_auc],
    "gini": [best_valid_gini, test_gini]
})

final_results

## Final Model Summary

The final model was Logistic Regression with `C = 0.05` and a classification threshold of `0.15`.

The model performed consistently on validation and test data.

| Dataset | Precision | Recall | F1 | ROC AUC | Gini |
|---|---:|---:|---:|---:|---:|
| Validation | 0.2086 | 0.5730 | 0.3058 | 0.6848 | 0.3697 |
| Test | 0.2123 | 0.6302 | 0.3176 | 0.6991 | 0.3982 |

The test result is slightly better than the validation result, which suggests that the model generalizes well to unseen future data.

The model has low Precision but good Recall. This means it catches many bad-buy cars, but it also marks some good cars as risky. For this problem, this trade-off can be acceptable because missing a bad-buy car may be more costly than reviewing extra cars.

## Manual Implementation of Classification Metrics

To understand how classification metrics work, I implemented Precision, Recall, and F1 score manually.

These metrics are calculated from the confusion matrix values:

- True Positive: predicted bad buy and it is actually bad buy
- False Positive: predicted bad buy but it is actually not bad buy
- False Negative: predicted not bad buy but it is actually bad buy
- True Negative: predicted not bad buy and it is actually not bad buy

In [ ]:
def manual_precision(y_true, y_pred):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    
    tp = np.sum((y_true == 1) & (y_pred == 1))
    fp = np.sum((y_true == 0) & (y_pred == 1))
    
    if tp + fp == 0:
        return 0
    
    return tp / (tp + fp)


def manual_recall(y_true, y_pred):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    
    tp = np.sum((y_true == 1) & (y_pred == 1))
    fn = np.sum((y_true == 1) & (y_pred == 0))
    
    if tp + fn == 0:
        return 0
    
    return tp / (tp + fn)


def manual_f1(y_true, y_pred):
    precision = manual_precision(y_true, y_pred)
    recall = manual_recall(y_true, y_pred)
    
    if precision + recall == 0:
        return 0
    
    return 2 * precision * recall / (precision + recall)

In [ ]:
manual_precision_value = manual_precision(y_test, test_pred)
manual_recall_value = manual_recall(y_test, test_pred)
manual_f1_value = manual_f1(y_test, test_pred)

print("Manual Precision:", manual_precision_value)
print("Manual Recall:", manual_recall_value)
print("Manual F1:", manual_f1_value)

print("\nSklearn Precision:", test_precision)
print("Sklearn Recall:", test_recall)
print("Sklearn F1:", test_f1)

## Manual Metrics Validation

The manually implemented Precision, Recall, and F1 score produced the same results as sklearn.

This confirms that the custom metric functions are implemented correctly.

The final test results are:

- Manual Precision: `0.2123`
- Manual Recall: `0.6302`
- Manual F1 score: `0.3176`

These values match the sklearn metrics exactly.

In [ ]:
def manual_gini(y_true, y_proba):
    auc = roc_auc_score(y_true, y_proba)
    return 2 * auc - 1

manual_gini_value = manual_gini(y_test, test_proba)

print("Manual Gini:", manual_gini_value)
print("Sklearn-based Gini:", test_gini)

## Manual Gini Validation

The Gini score was calculated manually from ROC AUC using the formula:

`Gini = 2 × ROC AUC - 1`

The manual Gini result matched the previous sklearn-based result exactly:

- Manual Gini: `0.3982`
- Sklearn-based Gini: `0.3982`

This confirms that the Gini calculation is correct.

In [ ]:
from sklearn.metrics import average_precision_score

test_auc_pr = average_precision_score(y_test, test_proba)

print("Test AUC PR:", test_auc_pr)

## AUC PR Result

AUC PR was calculated for the final Logistic Regression model on the test set.

The model achieved:

- AUC PR: `0.2746`

AUC PR is useful here because the dataset is imbalanced and the bad-buy class is much smaller than the good-buy class.

This score shows how well the model balances Precision and Recall across different thresholds.

In [ ]:
final_results = pd.DataFrame({
    "dataset": ["Validation", "Test"],
    "precision": [final_precision, test_precision],
    "recall": [final_recall, test_recall],
    "f1": [final_f1, test_f1],
    "roc_auc": [best_valid_auc, test_auc],
    "gini": [best_valid_gini, test_gini],
    "auc_pr": [average_precision_score(y_valid, best_valid_proba), test_auc_pr]
})

final_results

## Final Metrics Table

The final results table includes both threshold-based metrics and probability-based metrics.

Threshold-based metrics:
- Precision
- Recall
- F1 score

Probability-based metrics:
- ROC AUC
- Gini
- AUC PR

The final model performed slightly better on the test set than on the validation set.

| Dataset | Precision | Recall | F1 | ROC AUC | Gini | AUC PR |
|---|---:|---:|---:|---:|---:|---:|
| Validation | 0.2086 | 0.5730 | 0.3058 | 0.6848 | 0.3697 | 0.2269 |
| Test | 0.2123 | 0.6302 | 0.3176 | 0.6991 | 0.3982 | 0.2746 |

The model has strong Recall compared with Precision, meaning it detects many bad-buy cars but also produces false positives. This trade-off is acceptable if the business goal is to reduce the risk of missing bad-buy cars.

## Manual KNN Implementation

A simple KNN classifier will be implemented from scratch.

The model works by:
- storing the training data
- calculating distances between validation samples and training samples
- selecting the nearest `k` neighbors
- using the average label of the neighbors as the probability of class `1`

To keep the implementation simple and fast, only numeric features will be used in this custom version.

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer

# Use only numeric features for the manual KNN implementation
X_train_num = X_train[numeric_features].copy()
X_valid_num = X_valid[numeric_features].copy()

# Handle missing values
num_imputer = SimpleImputer(strategy="median")

X_train_num_imputed = num_imputer.fit_transform(X_train_num)
X_valid_num_imputed = num_imputer.transform(X_valid_num)

# Scale features because KNN depends on distance
scaler = StandardScaler()

X_train_num_scaled = scaler.fit_transform(X_train_num_imputed)
X_valid_num_scaled = scaler.transform(X_valid_num_imputed)

y_train_array = y_train.values

In [ ]:
class ManualKNNClassifier:
    def __init__(self, k=5):
        self.k = k
    
    def fit(self, X, y):
        self.X_train = np.array(X)
        self.y_train = np.array(y)
        return self
    
    def predict_proba(self, X):
        X = np.array(X)
        probabilities = []
        
        for sample in X:
            distances = np.sqrt(np.sum((self.X_train - sample) ** 2, axis=1))
            nearest_indices = np.argsort(distances)[:self.k]
            nearest_labels = self.y_train[nearest_indices]
            
            probability_class_1 = np.mean(nearest_labels)
            probabilities.append(probability_class_1)
        
        probabilities = np.array(probabilities)
        
        return np.column_stack([
            1 - probabilities,
            probabilities
        ])
    
    def predict(self, X, threshold=0.5):
        probabilities = self.predict_proba(X)[:, 1]
        return (probabilities >= threshold).astype(int)

In [ ]:
manual_knn = ManualKNNClassifier(k=5)

manual_knn.fit(X_train_num_scaled, y_train_array)

X_valid_sample = X_valid_num_scaled[:1000]
y_valid_sample = y_valid.values[:1000]

manual_knn_proba = manual_knn.predict_proba(X_valid_sample)[:, 1]
manual_knn_pred = manual_knn.predict(X_valid_sample)

manual_knn_auc = roc_auc_score(y_valid_sample, manual_knn_proba)
manual_knn_gini = 2 * manual_knn_auc - 1
manual_knn_precision = manual_precision(y_valid_sample, manual_knn_pred)
manual_knn_recall = manual_recall(y_valid_sample, manual_knn_pred)
manual_knn_f1 = manual_f1(y_valid_sample, manual_knn_pred)

print("Manual KNN ROC AUC:", manual_knn_auc)
print("Manual KNN Gini:", manual_knn_gini)
print("Manual KNN Precision:", manual_knn_precision)
print("Manual KNN Recall:", manual_knn_recall)
print("Manual KNN F1:", manual_knn_f1)

## Manual KNN Result

The manual KNN classifier was tested on a validation sample of 1,000 rows.

The model achieved:

- ROC AUC: `0.6155`
- Gini score: `0.2310`
- Precision: `0.2857`
- Recall: `0.0380`
- F1 score: `0.0670`

The manual KNN implementation works correctly and produces a positive Gini score.

Only numeric features were used in this custom version to keep the implementation simple and fast. This is why the result is not directly comparable to the full sklearn KNN model, which used both numeric and categorical features.

## Manual Gaussian Naive Bayes Implementation

A simple Gaussian Naive Bayes classifier will be implemented from scratch.

The model estimates the mean and variance of each numeric feature for each class.

During prediction, it calculates how likely each sample belongs to class `0` or class `1`, then returns the probability of `IsBadBuy = 1`.

Only numeric features will be used in this manual implementation.

In [ ]:
class ManualGaussianNB:
    def fit(self, X, y):
        X = np.array(X)
        y = np.array(y)
        
        self.classes = np.unique(y)
        self.mean = {}
        self.var = {}
        self.prior = {}
        
        for cls in self.classes:
            X_cls = X[y == cls]
            self.mean[cls] = X_cls.mean(axis=0)
            self.var[cls] = X_cls.var(axis=0) + 1e-9
            self.prior[cls] = X_cls.shape[0] / X.shape[0]
        
        return self
    
    def _log_gaussian_probability(self, X, cls):
        mean = self.mean[cls]
        var = self.var[cls]
        
        return -0.5 * np.sum(
            np.log(2 * np.pi * var) + ((X - mean) ** 2) / var,
            axis=1
        )
    
    def predict_proba(self, X):
        X = np.array(X)
        log_probs = []
        
        for cls in self.classes:
            log_prior = np.log(self.prior[cls])
            log_likelihood = self._log_gaussian_probability(X, cls)
            log_probs.append(log_prior + log_likelihood)
        
        log_probs = np.vstack(log_probs).T
        
        # stable softmax
        log_probs_shifted = log_probs - np.max(log_probs, axis=1, keepdims=True)
        probs = np.exp(log_probs_shifted)
        probs = probs / probs.sum(axis=1, keepdims=True)
        
        return probs
    
    def predict(self, X, threshold=0.5):
        probabilities = self.predict_proba(X)[:, 1]
        return (probabilities >= threshold).astype(int)

In [ ]:
manual_nb = ManualGaussianNB()

manual_nb.fit(X_train_num_scaled, y_train_array)

manual_nb_proba = manual_nb.predict_proba(X_valid_num_scaled)[:, 1]
manual_nb_pred = manual_nb.predict(X_valid_num_scaled)

manual_nb_auc = roc_auc_score(y_valid, manual_nb_proba)
manual_nb_gini = 2 * manual_nb_auc - 1
manual_nb_precision = manual_precision(y_valid, manual_nb_pred)
manual_nb_recall = manual_recall(y_valid, manual_nb_pred)
manual_nb_f1 = manual_f1(y_valid, manual_nb_pred)

print("Manual GaussianNB ROC AUC:", manual_nb_auc)
print("Manual GaussianNB Gini:", manual_nb_gini)
print("Manual GaussianNB Precision:", manual_nb_precision)
print("Manual GaussianNB Recall:", manual_nb_recall)
print("Manual GaussianNB F1:", manual_nb_f1)

## Manual Gaussian Naive Bayes Result

The manual Gaussian Naive Bayes classifier was evaluated on the validation set using numeric features only.

The model achieved:

- ROC AUC: `0.6494`
- Gini score: `0.2987`
- Precision: `0.2291`
- Recall: `0.2521`
- F1 score: `0.2400`

The manual implementation produced a positive Gini score and performed reasonably well.

This result is better than the previous Gaussian Naive Bayes baseline, likely because this manual version used only numeric features and avoided high-dimensional one-hot encoded categorical features.
    

## Manual Logistic Regression Implementation

A simple Logistic Regression classifier will be implemented from scratch using gradient descent.

The model learns weights for the input features and uses the sigmoid function to convert the linear output into a probability between `0` and `1`.

Only numeric features will be used in this manual implementation to keep the training process simple and stable.

In [ ]:
class ManualLogisticRegression:
    def __init__(self, learning_rate=0.01, n_iterations=1000):
        self.learning_rate = learning_rate
        self.n_iterations = n_iterations
    
    def _sigmoid(self, z):
        z = np.clip(z, -500, 500)
        return 1 / (1 + np.exp(-z))
    
    def fit(self, X, y):
        X = np.array(X)
        y = np.array(y)
        
        n_samples, n_features = X.shape
        
        self.weights = np.zeros(n_features)
        self.bias = 0
        
        for _ in range(self.n_iterations):
            linear_output = np.dot(X, self.weights) + self.bias
            probabilities = self._sigmoid(linear_output)
            
            dw = (1 / n_samples) * np.dot(X.T, (probabilities - y))
            db = (1 / n_samples) * np.sum(probabilities - y)
            
            self.weights -= self.learning_rate * dw
            self.bias -= self.learning_rate * db
        
        return self
    
    def predict_proba(self, X):
        X = np.array(X)
        linear_output = np.dot(X, self.weights) + self.bias
        probabilities = self._sigmoid(linear_output)
        
        return np.column_stack([
            1 - probabilities,
            probabilities
        ])
    
    def predict(self, X, threshold=0.5):
        probabilities = self.predict_proba(X)[:, 1]
        return (probabilities >= threshold).astype(int)

In [ ]:
manual_lr = ManualLogisticRegression(
    learning_rate=0.01,
    n_iterations=2000
)

manual_lr.fit(X_train_num_scaled, y_train_array)

manual_lr_proba = manual_lr.predict_proba(X_valid_num_scaled)[:, 1]
manual_lr_pred = manual_lr.predict(X_valid_num_scaled)

manual_lr_auc = roc_auc_score(y_valid, manual_lr_proba)
manual_lr_gini = 2 * manual_lr_auc - 1
manual_lr_precision = manual_precision(y_valid, manual_lr_pred)
manual_lr_recall = manual_recall(y_valid, manual_lr_pred)
manual_lr_f1 = manual_f1(y_valid, manual_lr_pred)

print("Manual Logistic Regression ROC AUC:", manual_lr_auc)
print("Manual Logistic Regression Gini:", manual_lr_gini)
print("Manual Logistic Regression Precision:", manual_lr_precision)
print("Manual Logistic Regression Recall:", manual_lr_recall)
print("Manual Logistic Regression F1:", manual_lr_f1)

## Manual Logistic Regression Result

The manual Logistic Regression model was evaluated on the validation set using numeric features only.

The model achieved:

- ROC AUC: `0.6567`
- Gini score: `0.3135`
- Precision: `0.0000`
- Recall: `0.0000`
- F1 score: `0.0000`

The positive ROC AUC and Gini score show that the model can rank bad-buy cars reasonably well.

However, Precision, Recall, and F1 are all zero because the default threshold of `0.5` does not classify any samples as bad buys.

This confirms that threshold choice is important for imbalanced classification problems.

In [ ]:
manual_lr_threshold_results = []

for threshold in np.arange(0.05, 0.51, 0.05):
    manual_lr_pred_threshold = (manual_lr_proba >= threshold).astype(int)
    
    precision_t = manual_precision(y_valid, manual_lr_pred_threshold)
    recall_t = manual_recall(y_valid, manual_lr_pred_threshold)
    f1_t = manual_f1(y_valid, manual_lr_pred_threshold)
    
    manual_lr_threshold_results.append({
        "threshold": threshold,
        "precision": precision_t,
        "recall": recall_t,
        "f1": f1_t
    })

manual_lr_threshold_results_df = pd.DataFrame(manual_lr_threshold_results)
manual_lr_threshold_results_df.sort_values("f1", ascending=False)

## Manual Logistic Regression Threshold Tuning Result

Different thresholds were tested for the manual Logistic Regression model.

The best F1 score was achieved with:

- Threshold: `0.15`
- Precision: `0.1877`
- Recall: `0.5552`
- F1 score: `0.2805`

Using a lower threshold improved the model's ability to detect bad-buy cars.

This confirms that threshold tuning is important when working with imbalanced binary classification data.

In [ ]:
manual_models_results = pd.DataFrame({
    "model": [
        "Manual KNN",
        "Manual Gaussian Naive Bayes",
        "Manual Logistic Regression"
    ],
    "roc_auc": [
        manual_knn_auc,
        manual_nb_auc,
        manual_lr_auc
    ],
    "gini": [
        manual_knn_gini,
        manual_nb_gini,
        manual_lr_gini
    ],
    "precision": [
        manual_knn_precision,
        manual_nb_precision,
        0.187652
    ],
    "recall": [
        manual_knn_recall,
        manual_nb_recall,
        0.555247
    ],
    "f1": [
        manual_knn_f1,
        manual_nb_f1,
        0.280505
    ]
})

manual_models_results.sort_values("gini", ascending=False)

## Manual Models Comparison Result

The custom models were compared using ROC AUC, Gini, Precision, Recall, and F1 score.

| Model | ROC AUC | Gini | Precision | Recall | F1 |
|---|---:|---:|---:|---:|---:|
| Manual Logistic Regression | 0.6567 | 0.3135 | 0.1877 | 0.5552 | 0.2805 |
| Manual Gaussian Naive Bayes | 0.6494 | 0.2987 | 0.2291 | 0.2521 | 0.2400 |
| Manual KNN | 0.6155 | 0.2310 | 0.2857 | 0.0380 | 0.0670 |

Manual Logistic Regression achieved the best overall result because it had the highest ROC AUC, Gini score, Recall, and F1 score.

Manual Gaussian Naive Bayes performed reasonably well, but it had lower Recall and F1.

Manual KNN had the highest Precision, but its Recall was very low, meaning it missed most bad-buy cars.

## Best Model

The best model is Logistic Regression with `C = 0.05`.

It achieved the highest validation Gini score among the tested sklearn models and also performed well on the final test set.

This model was selected because it gave the best balance between ranking quality, measured by Gini and ROC AUC, and bad-buy detection, measured by Recall and F1 score.

## Final Conclusion

This project focused on solving a binary classification problem using the `IsBadBuy` target.

The goal was to predict whether a car is a bad buy (`1`) or not (`0`) using vehicle, auction, price, and warranty-related features.

The dataset was split by purchase date into training, validation, and test sets to avoid data leakage. The model was trained on older purchases and evaluated on newer unseen purchases.

Three sklearn classification models were tested:

- Logistic Regression
- Gaussian Naive Bayes
- KNN

Logistic Regression was the best sklearn model. After tuning, the best model used `C = 0.05` and a classification threshold of `0.15`.

| Metric | Test Score |
|---|---:|
| Precision | 0.2123 |
| Recall | 0.6302 |
| F1 score | 0.3176 |
| ROC AUC | 0.6991 |
| Gini score | 0.3982 |
| AUC PR | 0.2746 |

The Gini score is above the required minimum, so the final model meets the project requirement.

The model has stronger Recall than Precision. This means it detects many bad-buy cars, but it also marks some good cars as risky. For this task, this trade-off can be acceptable because missing a bad-buy car may be more costly than reviewing extra cars.

Custom versions of KNN, Gaussian Naive Bayes, and Logistic Regression were also implemented. Among the manual models, Logistic Regression achieved the best overall performance.

Overall, Logistic Regression was the most stable and effective model for this dataset.

## Limitations and Possible Improvements

The final model meets the project requirement, but there are still some limitations.

One limitation is the low Precision score. The model detects many bad-buy cars, but it also creates false positives, which may require extra manual review.

The feature engineering step did not improve the validation Gini score. Better features could be created from vehicle history, auction behavior, seller information, or price changes over time.

KNN was slower and less effective than Logistic Regression, especially because the dataset contains many rows and categorical features.

Gaussian Naive Bayes performed poorly in the sklearn baseline, probably because its assumptions were too simple for this dataset.

Possible improvements:

- test more advanced models such as Random Forest, Gradient Boosting, or LightGBM
- tune more hyperparameters
- improve feature engineering
- test different classification thresholds based on business cost
- handle categorical features with more advanced encoding methods
- evaluate the model using cost-based metrics if false-positive and false-negative costs are known

Overall, Logistic Regression is a strong and stable baseline, but more advanced models could potentially improve the final performance.